# Phase 4: Bilateral Market-Making vs Baseline Comparison

This notebook trains a bilateral market-making RL agent and compares it against a simple baseline.

**Workflow**:
1. Setup environment and dependencies
2. Clone/pull latest code from repository
3. Implement SymmetricFixedSpread baseline agent
4. Train bilateral RL agent (200 iterations, quick config)
5. Evaluate bilateral agent (1000 episodes)
6. Evaluate baseline agent (1000 episodes)
7. Compare metrics and visualize results

**Runtime**: ~30-40 minutes (GPU)

---

## Step 0: Clear cache and setup repository


In [1]:
!git clone https://github.com/SalmanSattar24/rtle_parallelized.git /content/rtle_parallelized
%cd /content/rtle_parallelized

fatal: destination path '/content/rtle_parallelized' already exists and is not an empty directory.
/content/rtle_parallelized


## Step 1: Setup Environment

In [2]:
# Install dependencies (Colab)
import sys
import subprocess

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("[INFO] Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("[INFO] Running locally")

# Install dependencies if in Colab
if IN_COLAB:
    print("\n[INSTALL] Installing PyTorch and dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "gymnasium", "tensorboard", "tyro"])
    print("[OK] Dependencies installed")
else:
    print("[CHECK] Verifying dependencies...")
    try:
        import torch
        import gymnasium
        import tensorboard
        print("[OK] All dependencies present")
    except ImportError as e:
        print(f"[WARNING] Missing dependency: {e}")

[INFO] Running in Google Colab

[INSTALL] Installing PyTorch and dependencies...
[OK] Dependencies installed


## Step 2: Clone/Pull Repository

In [3]:
import os
import subprocess
import shutil

# Setup directory paths
if IN_COLAB:
    # Clone to /content in Colab
    repo_dir = "/content/rtle_parallelized"
    if os.path.exists(repo_dir):
        print(f"[PULL] Updating existing repo at {repo_dir}")
        os.chdir(repo_dir)
        subprocess.run(["git", "pull"], check=True, capture_output=True)
    else:
        print(f"[CLONE] Cloning repository to {repo_dir}")
        subprocess.run(
            ["git", "clone", "https://github.com/SalmanSattar24/rtle_parallelized.git", repo_dir],
            check=True,
            capture_output=True
        )
else:
    # Local path
    repo_dir = "C:/All-Code/CSCI-566/rtle_parallelized"
    print(f"[INFO] Using local repository at {repo_dir}")
    if os.path.exists(os.path.join(repo_dir, ".git")):
        os.chdir(repo_dir)
        print("[PULL] Updating repository...")
        subprocess.run(["git", "pull"], capture_output=True)

os.chdir(repo_dir)
print(f"\n[OK] Working directory: {os.getcwd()}")
print(f"[VERIFY] Repository structure:")
for folder in ["simulation", "rl_files", "tests", "limit_order_book"]:
    path = os.path.join(repo_dir, folder)
    print(f"  {'[OK]' if os.path.exists(path) else '[MISS]'} {folder}/")

[PULL] Updating existing repo at /content/rtle_parallelized

[OK] Working directory: /content/rtle_parallelized
[VERIFY] Repository structure:
  [OK] simulation/
  [OK] rl_files/
  [OK] tests/
  [OK] limit_order_book/


## Step 3.5: Update Repository to Latest Version

In [4]:
print("=" * 70)
print("STEP 3.5: Updating Repository to Latest Version")
print("=" * 70)

print("\nFetching latest commits from GitHub...")
import subprocess
subprocess.run(["git", "pull", "origin", "master"], cwd=repo_dir, check=True)

print("\nâœ“ Repository updated!")
print("=" * 70)


STEP 3.5: Updating Repository to Latest Version

Fetching latest commits from GitHub...

âœ“ Repository updated!


In [5]:
# Sync guard: verify Colab/runtime commit before expensive runs
import subprocess
from pathlib import Path

repo_path = Path(repo_dir)
try:
    current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_path).decode().strip()
    current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_path).decode().strip()
    print(f"[GIT] Branch: {current_branch}")
    print(f"[GIT] Commit: {current_commit}")
except Exception as e:
    current_commit = None
    print(f"[WARN] Could not read git commit: {e}")

# Optional strict pin: set this to the commit you expect after push
EXPECTED_COMMIT = None  # e.g., "a338e21"

if EXPECTED_COMMIT is not None and current_commit is not None:
    assert current_commit == EXPECTED_COMMIT, (
        f"Commit mismatch! expected={EXPECTED_COMMIT}, got={current_commit}. "
        "Run repo update cell before training/eval."
    )
    print("[OK] Commit hash matches expected version")
else:
    print("[INFO] EXPECTED_COMMIT is not set; skipping strict commit assertion")

[GIT] Branch: master
[GIT] Commit: ce5c751
[INFO] EXPECTED_COMMIT is not set; skipping strict commit assertion


In [6]:
import numpy as np

print("=" * 70)
print("PHASE 6: AVELLANEDA-STOIKOV (AS) BASELINE CALCULATION")
print("=" * 70)

def compute_as_quotes(mid_price, inventory, time_left, sigma=2.0, gamma=0.1, k=1.5):
    # reservation_price = mid_price - q * gamma * sigma^2 * (T - t)
    # spread = (2 / gamma) * ln(1 + gamma / k) + gamma * sigma^2 * (T - t)
    
    res_price = mid_price - inventory * gamma * (sigma**2) * time_left
    spread = (2 / gamma) * np.log(1 + gamma / k) + gamma * (sigma**2) * time_left
    
    bid_price = np.floor(res_price - spread / 2)
    ask_price = np.ceil(res_price + spread / 2)
    
    return bid_price, ask_price

# Example AS quotes
mid = 1000
inv = 5
t_left = 0.5
b, a = compute_as_quotes(mid, inv, t_left)
print(f"[AS TEST] Mid: {mid} | Inv: {inv} | Bid: {b} | Ask: {a} | Spread: {a-b}")


PHASE 6: AVELLANEDA-STOIKOV (AS) BASELINE CALCULATION
[AS TEST] Mid: 1000 | Inv: 5 | Bid: 998.0 | Ask: 1000.0 | Spread: 2.0


## Step 4: Import Libraries and Setup Paths

In [7]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import time
from typing import Optional, Dict, List, Tuple
import matplotlib.pyplot as plt

# Add paths for imports
sys.path.insert(0, repo_dir)
sys.path.insert(0, os.path.join(repo_dir, "simulation"))
sys.path.insert(0, os.path.join(repo_dir, "rl_files"))
sys.path.insert(0, os.path.join(repo_dir, "limit_order_book"))

# Import core modules
from simulation.market_gym import Market
from rl_files.actor_critic import BilateralAgentLogisticNormal

print("[OK] All imports successful")
print(f"[INFO] Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
print("[OK] Random seeds set")

[OK] All imports successful
[INFO] Using device: cuda
[OK] Random seeds set


In [8]:
print("="*70)
print("FORCE FRESH REPO LOAD (clear cached imports)")
print("="*70)

# Force reimport of all modules - clear cached modules
import sys
to_remove = [key for key in sys.modules if 'simulation' in key or 'rl_files' in key or 'limit_order_book' in key]
for key in to_remove:
    del sys.modules[key]

print("[OK] Cleared cached modules")
print("="*70 + "\n")

FORCE FRESH REPO LOAD (clear cached imports)
[OK] Cleared cached modules



## Step 4: Implement SymmetricFixedSpread Baseline Agent

In [9]:
from typing import Tuple

class SymmetricFixedSpreadAgent:
    """
    Baseline market-making agent:
    - Posts 1 lot at best bid (passive buy)
    - Posts 1 lot at best ask (passive sell)
    - Returns TUPLE format to match environment expectations
    - Simple, static strategy
    """

    def __init__(self, action_space_dim: int = 7):
        self.action_space_dim = action_space_dim
        # Action allocation:
        # [market%, L1%, L2%, L3%, L4%, L5%, inactive%]
        # For baseline: 0% market, 100% L1 (1 lot at best)
        self.action = np.zeros(action_space_dim)
        self.action[1] = 1.0  # Place 100% at level 1 (best bid/ask)

    def get_action(self, obs: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Return fixed action tuple (bid_action, ask_action) regardless of observation.
        
        Both bid and ask use the same allocation (symmetric fixed spread).
        Environment expects tuple of two 7-dim arrays.
        """
        return self.action.copy(), self.action.copy()

print("[OK] SymmetricFixedSpreadAgent defined (returns tuple format)")

[OK] SymmetricFixedSpreadAgent defined (returns tuple format)


## Step 5: Configuration

In [10]:
# SOTA: Phase 6 Configuration (Cheridito & Weiss 2026 Alignment)
TRAIN_CONFIG = {
    'market_env': 'strategic', # Noise + Tactical + Strategic Traders
    'execution_agent': 'rl_agent',
    'volume': 20, # Reduced to 20 for 10-lot buffer to 30 limit
    'seed': 42,
    'terminal_time': 500,
    'time_delta': 50,
    'drop_feature': None,
    'inventory_max': 30, 
    'penalty_weight': 0.005,
}

TRAIN_PARAMS = {
    'num_iterations': 200,
    'num_steps': 10,
    'batch_size': 10,
    'learning_rate': 5e-4,
    'entropy_coef': 0.05,
    'vf_coef': 0.5,
    'gamma': 1.0,
    'gae_lambda': 1.0,
}

EVAL_CONFIG = {
    'market_env': 'strategic',
    'execution_agent': 'rl_agent',
    'volume': 20, # Reduced to 20
    'seed': 100,
    'terminal_time': 500,
    'time_delta': 50,
    'drop_feature': None,
    'inventory_max': 30,
    'penalty_weight': 0.005,
}

EVAL_EPISODES = 1000

print("[OK] Phase 6 Configuration loaded (Volume: 20, Limit: 30)")


[OK] Phase 6 Configuration loaded
     Environment: strategic
     Inventory Max: 30


## Step 6: Create Market Environment and Agents

In [11]:
print("[SETUP] Creating market environment...")
market_env = Market(TRAIN_CONFIG)
obs, info = market_env.reset(seed=42)

print(f"[INFO] Observation shape: {obs.shape}")
print(f"[INFO] Action space: {market_env.action_space.shape}")

# Create a simple wrapper so BilateralAgentLogisticNormal can work with Market
class EnvWrapper:
    def __init__(self, env):
        self.env = env
        # Add single_observation_space and single_action_space for agent init
        self.single_observation_space = env.observation_space
        self.single_action_space = env.action_space
    
    # Proxy methods to underlying environment
    def reset(self, seed=None):
        return self.env.reset(seed=seed)
    
    def step(self, action):
        return self.env.step(action)

market = EnvWrapper(market_env)

print("\n[SETUP] Creating bilateral RL agent...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bilateral_agent = BilateralAgentLogisticNormal(market).to(device)
print(f"[OK] Bilateral agent on {device}")

print("\n[SETUP] Creating baseline agent...")
baseline_agent = SymmetricFixedSpreadAgent(market_env.action_space.shape[0])
print(f"[OK] Baseline agent ready")

print("\n[SETUP] All agents created successfully")

[SETUP] Creating market environment...
[INFO] Observation shape: (110,)
[INFO] Action space: (7,)

[SETUP] Creating bilateral RL agent...
[OK] Bilateral agent on cuda

[SETUP] Creating baseline agent...
[OK] Baseline agent ready

[SETUP] All agents created successfully


## Step 7: Train Bilateral Agent


## Step 7A: Minimal Training 

In [12]:
print("="*70)
print("STEP 7A: QUICK TRAINING TEST (minimal version)")
print("="*70)
print()

# Test config: tiny episodes for speed
QUICK_CONFIG = {
    'market_env': 'noise',
    'execution_agent': 'rl_agent',
    'volume': 40,
    'seed': 42,
    'terminal_time': 50,  # 10x shorter episodes
    'time_delta': 50,
    'drop_feature': None,
    'inventory_max': 10,
    'penalty_weight': 1.0,
}

QUICK_PARAMS = {
    'num_iterations': 5,
    'num_steps': 2,
    'learning_rate': 5e-4,
    'entropy_coef': 0.05,
    'vf_coef': 0.5,
    'gamma': 1.0,
}

print(f"[CONFIG] terminal_time={QUICK_CONFIG['terminal_time']} | iterations={QUICK_PARAMS['num_iterations']} | steps={QUICK_PARAMS['num_steps']}")
print()

market_test = Market(QUICK_CONFIG)
optimizer = torch.optim.Adam(bilateral_agent.parameters(), lr=QUICK_PARAMS['learning_rate'])
training_returns = []
start_time = time.time()

for iteration in range(QUICK_PARAMS['num_iterations']):
    print(f"[{iteration+1:2d}/{QUICK_PARAMS['num_iterations']}] ", end='', flush=True)
    
    batch_states, batch_actions, batch_rewards, batch_values = [], [], [], []
    
    for step in range(QUICK_PARAMS['num_steps']):
        obs, _ = market_test.reset(seed=42 + iteration + step)
        ep_return = 0
        timesteps = 0
        
        while True:
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                actions, log_prob, entropy, value = bilateral_agent.get_action_and_value(obs_tensor)
            
            batch_states.append(obs_tensor.detach())
            batch_actions.append(actions)
            batch_values.append(value.detach())
            
            bid_action, ask_action = actions
            env_action = (bid_action[0].cpu().numpy(), ask_action[0].cpu().numpy())
            obs, reward, terminated, truncated, info = market_test.step(env_action)
            batch_rewards.append(reward)
            ep_return += reward
            timesteps += 1
            
            if terminated:
                break
        
        training_returns.append(ep_return)
    
    # Compute and apply loss
    batch_returns = []
    cumulative_return = 0
    for i in range(len(batch_rewards)-1, -1, -1):
        cumulative_return = batch_rewards[i] + QUICK_PARAMS['gamma'] * cumulative_return
        batch_returns.insert(0, cumulative_return)
    
    optimizer.zero_grad()
    total_loss = 0
    for i in range(min(len(batch_states), len(batch_returns))):
        _, log_prob, entropy, value = bilateral_agent.get_action_and_value(batch_states[i], action=batch_actions[i])
        actor_loss = -(log_prob * (batch_returns[i] - batch_values[i].item()))
        value_loss = 0.5 * (value.squeeze() - batch_returns[i]) ** 2
        total_loss = total_loss + actor_loss + value_loss - 0.05 * entropy
    
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(bilateral_agent.parameters(), max_norm=0.5)
    optimizer.step()
    
    elapsed = time.time() - start_time
    avg_ret = np.mean(training_returns[-QUICK_PARAMS['num_steps']:])
    print(f"Return: {avg_ret:7.2f} | Time: {elapsed:5.1f}s", flush=True)

total_time = time.time() - start_time
print()
print(f"[OK] Test complete in {total_time:.1f}s")
print(f"[ESTIMATE] Full training (200 iter × 10 steps): ~{(200*10)/(QUICK_PARAMS['num_iterations']*QUICK_PARAMS['num_steps'])*total_time/60:.0f} minutes")
print("="*70)
print("\nNext: Run cell 19 (STEP 7) for full training with terminal_time=500")
print("="*70 + "\n")

STEP 7A: QUICK TRAINING TEST (minimal version)

[CONFIG] terminal_time=50 | iterations=5 | steps=2

[ 1/5] Return: -561.95 | Time:   0.8s
[ 2/5] Return: -840.01 | Time:   1.3s
[ 3/5] Return: -1508.04 | Time:   1.7s
[ 4/5] Return: -401.51 | Time:   2.2s
[ 5/5] Return: -222.50 | Time:   2.7s

[OK] Test complete in 2.7s
[ESTIMATE] Full training (200 iter × 10 steps): ~9 minutes

Next: Run cell 19 (STEP 7) for full training with terminal_time=500



In [13]:
import torch

def project_action_risk(a, current_inv, side="bid", inventory_max=30.0):
    """
    Phase 6 Hard Quota Rule: Ensures orders never exceed remaining inventory room.
    """
    x = torch.clamp(a, min=1e-8)
    
    # 1. Calculate room relative to death-limit (30)
    # current_inv = (net_inventory)
    if side == "bid":
        # Bid = Buying. Room to +30.
        room = max(0.0, inventory_max - current_inv)
    else:
        # Ask = Selling. Room to -30.
        room = max(0.0, inventory_max + current_inv)
        
    # 2. Dynamic Order Cap: Limit per-step impact to 5 lots (25% of 20 total)
    order_cap = min(room, 5.0)
    active_mass_limit = order_cap / 20.0
    
    # 3. Apply Limit to Level 0 (Market) and Levels 1-5 (Limit)
    active_mass = torch.sum(x[..., :-1], dim=-1, keepdim=True)
    scale = torch.minimum(torch.ones_like(active_mass), active_mass_limit / (active_mass + 1e-8))
    
    x[..., :-1] *= scale
    
    # 4. Final Re-normalization: All remaining mass -> Inactive (index 6)
    total_scaled = torch.sum(x, dim=-1, keepdim=True)
    x[..., -1] += (1.0 - total_scaled).clamp(min=0)
    
    return x / (x.sum(dim=-1, keepdim=True) + 1e-8)


STEP 7: TRAIN BILATERAL AGENT (PPO + variance control v4)

[  1/150]   8.9s | Total:    8.9s | Avg20:   -54.96 | Loss:  39.7946 | Ent: 0.1000 | Var: 1.000 | Val120:      nan±   nan | CVaR5:      nan | Out<−200:    nan | Score:      nan
[  2/150]   8.5s | Total:   17.4s | Avg20:   -53.93 | Loss:  38.3239 | Ent: 0.0995 | Var: 0.995 | Val120:      nan±   nan | CVaR5:      nan | Out<−200:    nan | Score:      nan
[  3/150]   9.2s | Total:   26.6s | Avg20:   -53.19 | Loss:  37.4886 | Ent: 0.0989 | Var: 0.990 | Val120:      nan±   nan | CVaR5:      nan | Out<−200:    nan | Score:      nan
[  4/150]   9.1s | Total:   35.7s | Avg20:   -54.23 | Loss:  39.6788 | Ent: 0.0984 | Var: 0.986 | Val120:      nan±   nan | CVaR5:      nan | Out<−200:    nan | Score:      nan
[  5/150]   9.2s | Total:   44.9s | Avg20:   -53.75 | Loss:  38.6572 | Ent: 0.0979 | Var: 0.981 | Val120:      nan±   nan | CVaR5:      nan | Out<−200:    nan | Score:      nan
[ 15/150]  50.0s | Total:  175.4s | Avg20:   -54.84 | Lo

## Step 8: Evaluate Bilateral Agent


In [14]:
import torch

def project_action_risk(a, current_inv, side="bid", inventory_max=30.0):
    """
    Phase 6 Hard Quota Rule: Ensures orders never exceed remaining inventory room.
    """
    x = torch.clamp(a, min=1e-8)
    
    # 1. Calculate room relative to death-limit (30)
    # current_inv = (net_inventory)
    if side == "bid":
        # Bid = Buying. Room to +30.
        room = max(0.0, inventory_max - current_inv)
    else:
        # Ask = Selling. Room to -30.
        room = max(0.0, inventory_max + current_inv)
        
    # 2. Dynamic Order Cap: Limit per-step impact to 5 lots (25% of 20 total)
    order_cap = min(room, 5.0)
    active_mass_limit = order_cap / 20.0
    
    # 3. Apply Limit to Level 0 (Market) and Levels 1-5 (Limit)
    active_mass = torch.sum(x[..., :-1], dim=-1, keepdim=True)
    scale = torch.minimum(torch.ones_like(active_mass), active_mass_limit / (active_mass + 1e-8))
    
    x[..., :-1] *= scale
    
    # 4. Final Re-normalization: All remaining mass -> Inactive (index 6)
    total_scaled = torch.sum(x, dim=-1, keepdim=True)
    x[..., -1] += (1.0 - total_scaled).clamp(min=0)
    
    return x / (x.sum(dim=-1, keepdim=True) + 1e-8)


[EVAL] Evaluating bilateral agent on 1000 episodes...

[ 100/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 200/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 300/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 400/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 500/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 600/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 700/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 800/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 900/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[1000/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000

[OK] Bilateral evaluation complete in 1387.1s
[INFO] Mean return: -50.0000 +/- 0.0000
[INFO] Mean terminal inventory: 35.0380
[WARN] Near-zero return variance detected; verify market randomness/seeding.


## Step 9: Evaluate Baseline Agent


In [ ]:
print("[EVAL] Evaluating baseline agent on {} episodes...".format(EVAL_EPISODES))
print()

baseline_returns = []
baseline_inventories = []
baseline_times = []

start_eval = time.time()
base_seed = EVAL_CONFIG['seed']

for ep in range(EVAL_EPISODES):
    # Match bilateral eval seeding exactly for fair comparison
    ep_config = dict(EVAL_CONFIG)
    ep_config['seed'] = base_seed + ep

    eval_market_raw = Market(ep_config)
    eval_market = EnvWrapper(eval_market_raw)
    obs, _ = eval_market.reset()

    ep_return = 0
    ep_inventory = 0

    while True:
        bid_action, ask_action = baseline_agent.get_action(obs)
        env_action = (bid_action, ask_action)

        obs, reward, terminated, truncated, info = eval_market.step(env_action)
        ep_return += reward
        ep_inventory = info.get('net_inventory', 0)

        if terminated or truncated:
            break

    baseline_returns.append(ep_return)
    baseline_inventories.append(abs(ep_inventory))
    baseline_times.append(info.get('time', 0))

    if (ep + 1) % 100 == 0:
        elapsed = time.time() - start_eval
        print(f"[{ep+1:4d}/{EVAL_EPISODES}] Episodes/sec: {(ep+1)/elapsed:.1f} | Mean return: {np.mean(baseline_returns):.4f} | Std: {np.std(baseline_returns):.4f}")

eval_time = time.time() - start_eval
print(f"\n[OK] Baseline evaluation complete in {eval_time:.1f}s")
print(f"[INFO] Mean return: {np.mean(baseline_returns):.4f} +/- {np.std(baseline_returns):.4f}")
print(f"[INFO] Mean terminal inventory: {np.mean(baseline_inventories):.4f}")
if np.std(baseline_returns) < 1e-8:
    print("[WARN] Near-zero return variance detected; verify market randomness/seeding.")

[EVAL] Evaluating baseline agent on 1000 episodes...

[ 100/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 200/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000
[ 300/1000] Episodes/sec: 0.7 | Mean return: -50.0000 | Std: 0.0000


## Step 10: Compare Results


In [ ]:
# Compute statistics
bilateral_stats = {
    'mean_return': np.mean(bilateral_returns),
    'std_return': np.std(bilateral_returns),
    'min_return': np.min(bilateral_returns),
    'max_return': np.max(bilateral_returns),
    'mean_inventory': np.mean(bilateral_inventories),
}

baseline_stats = {
    'mean_return': np.mean(baseline_returns),
    'std_return': np.std(baseline_returns),
    'min_return': np.min(baseline_returns),
    'max_return': np.max(baseline_returns),
    'mean_inventory': np.mean(baseline_inventories),
}

# Improvement calculation
improvement = bilateral_stats['mean_return'] - baseline_stats['mean_return']
relative_improvement = (improvement / abs(baseline_stats['mean_return'])) * 100 if baseline_stats['mean_return'] != 0 else 0

print("\n" + "="*70)
print("COMPARISON: BILATERAL vs BASELINE")
print("="*70)
print()
print(f"{'Metric':<25} {'Bilateral':<20} {'Baseline':<20}")
print("-" * 70)
print(f"{'Mean Return':<25} {bilateral_stats['mean_return']:>8.4f} +/- {bilateral_stats['std_return']:6.4f}  {baseline_stats['mean_return']:>8.4f} +/- {baseline_stats['std_return']:6.4f}")
print(f"{'Min Return':<25} {bilateral_stats['min_return']:>18.4f}  {baseline_stats['min_return']:>18.4f}")
print(f"{'Max Return':<25} {bilateral_stats['max_return']:>18.4f}  {baseline_stats['max_return']:>18.4f}")
print(f"{'Terminal Inventory':<25} {bilateral_stats['mean_inventory']:>18.4f}  {baseline_stats['mean_inventory']:>18.4f}")
print("-" * 70)
print()
print(f"Improvement: {improvement:+.4f} ({relative_improvement:+.1f}%)")

if improvement > 0:
    print(f"\n[SUCCESS] Bilateral agent OUTPERFORMS baseline by {improvement:.4f} ({relative_improvement:.1f}%)")
else:
    print(f"\n[INFO] Baseline performs better by {-improvement:.4f} ({-relative_improvement:.1f}%)")

print("="*70)

## Step 11: Visualize Results

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bilateral RL Agent vs Baseline (SymmetricFixedSpread)', fontsize=16, fontweight='bold')

# Plot 1: Distribution of returns
ax = axes[0, 0]
ax.hist(bilateral_returns, bins=50, alpha=0.6, label='Bilateral RL', color='blue', edgecolor='black')
ax.hist(baseline_returns, bins=50, alpha=0.6, label='Baseline', color='orange', edgecolor='black')
ax.axvline(np.mean(bilateral_returns), color='blue', linestyle='--', linewidth=2, label=f"Bilateral Mean: {np.mean(bilateral_returns):.4f}")
ax.axvline(np.mean(baseline_returns), color='orange', linestyle='--', linewidth=2, label=f"Baseline Mean: {np.mean(baseline_returns):.4f}")
ax.set_xlabel('Episode Return')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Episode Returns')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Cumulative returns
ax = axes[0, 1]
ax.plot(np.cumsum(bilateral_returns), label='Bilateral RL', linewidth=2, color='blue')
ax.plot(np.cumsum(baseline_returns), label='Baseline', linewidth=2, color='orange')
ax.set_xlabel('Episode')
ax.set_ylabel('Cumulative Return')
ax.set_title('Cumulative Returns over 1000 Episodes')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Training curve
ax = axes[1, 0]
window = 20
training_ma = np.convolve(training_returns, np.ones(window)/window, mode='valid')
ax.plot(training_ma, label='Training Return (MA-20)', linewidth=2, color='purple')
ax.fill_between(range(len(training_ma)), 
                training_ma - np.std(training_returns[:len(training_ma)]), 
                training_ma + np.std(training_returns[:len(training_ma)]), 
                alpha=0.2, color='purple')
ax.set_xlabel('Training Iteration')
ax.set_ylabel('Episode Return')
ax.set_title('Bilateral Agent Training Curve')
ax.grid(True, alpha=0.3)

# Plot 4: Terminal inventory comparison
ax = axes[1, 1]
ax.boxplot([bilateral_inventories, baseline_inventories], 
           labels=['Bilateral', 'Baseline'],
           patch_artist=True)
ax.set_ylabel('Terminal Inventory (Absolute)')
ax.set_title('Terminal Inventory Management')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('phase4_comparison.png', dpi=100, bbox_inches='tight')
print("[OK] Plot saved as 'phase4_comparison.png'")
plt.show()

print("\nVisualization complete!")

## Step 12: Summary and Next Steps

In [ ]:
print("\n" + "="*70)
print("PHASE 4 SIMPLIFIED: EXECUTION COMPLETE")
print("="*70)
print()
print("RESULTS SUMMARY")
print("-" * 70)

train_iters_used = NUM_TRAIN_ITERS if 'NUM_TRAIN_ITERS' in globals() else TRAIN_PARAMS['num_iterations']
print(f"Training iterations:        {train_iters_used}")
print(f"Evaluation episodes:        {EVAL_EPISODES}")
print()
print(f"Bilateral RL Agent:")
print(f"  Mean return:              {bilateral_stats['mean_return']:>10.4f}")
print(f"  Std deviation:            {bilateral_stats['std_return']:>10.4f}")
print(f"  Terminal inventory:       {bilateral_stats['mean_inventory']:>10.4f}")
print()
print(f"Baseline Agent (SymmetricFixedSpread):")
print(f"  Mean return:              {baseline_stats['mean_return']:>10.4f}")
print(f"  Std deviation:            {baseline_stats['std_return']:>10.4f}")
print(f"  Terminal inventory:       {baseline_stats['mean_inventory']:>10.4f}")
print()
print(f"Performance Gap:            {improvement:>10.4f} ({relative_improvement:+.1f}%)")
print("-" * 70)
print()
if improvement > 0:
    print("[SUCCESS] Bilateral RL agent demonstrates improvement over simple baseline!")
    print()
    print("Key findings:")
    print(f"  1. RL agent achieves {improvement:.4f} better PnL per episode")
    print(f"  2. {'Better' if bilateral_stats['std_return'] < baseline_stats['std_return'] else 'Similar'} variance control")
    print(f"  3. {'Improved' if bilateral_stats['mean_inventory'] < baseline_stats['mean_inventory'] else 'Similar'} inventory management")
else:
    print(f"[INFO] Baseline performs better. This may indicate:")
    print(f"  1. Need for more training iterations (more than {train_iters_used})")
    print(f"  2. Hyperparameter tuning required")
    print(f"  3. Different environment complexity needed")

print()
print("PHASE 4 EXPANDED (Optional):")
print("-" * 70)
print("To extend this analysis:")
print("  1. Add more baseline agents (TWAP, Avellaneda-Stoikov)")
print("  2. Train on larger batch (400+ iterations)")
print("  3. Compare across multiple environments")
print("  4. Analyze learned trading strategy")
print("  5. Extract quote depth vs time-to-expiry patterns")
print()
print("PHASE 5: Documentation and Results")
print("-" * 70)
print("  1. Generate comparison tables")
print("  2. Create detailed analysis report")
print("  3. Document implementation findings")
print("  4. Package results for publication")
print()
print("="*70)
print("\n[OK] Phase 4 Simplified Complete!")

## Optional: Save Results

In [ ]:
# Save results to file
import json

results = {
    'bilateral': bilateral_stats,
    'baseline': baseline_stats,
    'improvement': {
        'absolute': float(improvement),
        'percentage': float(relative_improvement),
    },
    'config': {
        'train_iterations': TRAIN_PARAMS['num_iterations'],
        'eval_episodes': EVAL_EPISODES,
        'env_type': TRAIN_CONFIG['market_env'],
        'inventory_max': TRAIN_CONFIG['inventory_max'],
    },
}

with open('phase4_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("[OK] Results saved to 'phase4_results.json'")
print()
print("[INFO] You can download these files:")
print("  - phase4_comparison.png (visualization)")
print("  - phase4_results.json (raw data)")

In [ ]:
# Diagnostics snapshot (auto-generated)
import numpy as np

def summarize(name, arr):
    a = np.asarray(arr, dtype=float)
    return {
        'n': int(len(a)),
        'mean': float(np.mean(a)),
        'std': float(np.std(a)),
        'p05': float(np.percentile(a, 5)),
        'p50': float(np.percentile(a, 50)),
        'p95': float(np.percentile(a, 95)),
        'min': float(np.min(a)),
        'max': float(np.max(a)),
    }

print('=== TRAINING ===')
if 'training_returns' in globals() and len(training_returns) > 0:
    tr = np.asarray(training_returns, dtype=float)
    print('training_returns:', summarize('training_returns', tr))
    print('train_tail_mean_20:', float(np.mean(tr[-20:])))
else:
    print('training_returns not available')

if 'training_losses' in globals() and len(training_losses) > 0:
    tl = np.asarray(training_losses, dtype=float)
    print('training_losses:', summarize('training_losses', tl))
    print('loss_tail_mean_20:', float(np.mean(tl[-20:])))
else:
    print('training_losses not available')

print('\n=== EVALUATION ===')
if 'bilateral_returns' in globals() and 'baseline_returns' in globals():
    br = np.asarray(bilateral_returns, dtype=float)
    ba = np.asarray(baseline_returns, dtype=float)
    print('bilateral:', summarize('bilateral', br))
    print('baseline :', summarize('baseline', ba))
    print('improvement_abs:', float(np.mean(br) - np.mean(ba)))
    denom = abs(np.mean(ba)) if np.mean(ba) != 0 else 1.0
    print('improvement_pct:', float(100.0 * (np.mean(br) - np.mean(ba)) / denom))
    print('bilateral_outlier_rate_r<-500:', float(np.mean(br < -500)))
    print('baseline_outlier_rate_r<-500 :', float(np.mean(ba < -500)))
else:
    print('evaluation arrays not available')

if 'bilateral_inventories' in globals() and 'baseline_inventories' in globals():
    bi = np.asarray(bilateral_inventories, dtype=float)
    bj = np.asarray(baseline_inventories, dtype=float)
    print('bilateral_inventory:', summarize('bilateral_inventory', bi))
    print('baseline_inventory :', summarize('baseline_inventory', bj))

In [ ]:

print("=" * 70)
print("PHASE 6 VERIFICATION: CIRCUIT BREAKER 'CRASH TEST'")
print("=" * 70)

# 1. Setup a Stress-Test Agent that only BUYS to force an inventory breach
class ForceBreachAgent:
    def __init__(self, action_dim=7):
        self.buy_action = np.zeros(action_dim)
        self.buy_action[0] = 1.0 # 100% Market Buy
        self.inactive = np.zeros(action_dim)
        self.inactive[-1] = 1.0

    def get_action(self, obs):
        # Always Market Buy, Inactive on Ask
        return self.buy_action.copy(), self.inactive.copy()

# 2. Run Episode in 'Strategic' environment
test_cfg = copy.deepcopy(TRAIN_CONFIG)
test_cfg['inventory_max'] = 30 # absolute limit
test_cfg['market_env'] = 'strategic'

env = Market(test_cfg)
agent = ForceBreachAgent()
obs, info = env.reset(seed=42)

total_reward = 0
steps = 0
triggered = False

while True:
    action = agent.get_action(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    if info.get('circuit_breaker', False):
        triggered = True
        print(f"[SUCCESS] Circuit Breaker Triggered at Step {steps}!")
        print(f"[INFO] Final Inventory: {info['net_inventory']}")
        print(f"[INFO] Reward for this step: {reward}")
        break
        
    if terminated or truncated:
        break

print(f"\n[RESULT] Triggered: {triggered} | Total Reward: {total_reward:.2f}")
if triggered and -51.0 <= reward <= -49.0:
    print("VERIFICATION PASSED: Simulator correctly caps tail-risk at -50.0")
else:
    print("VERIFICATION FAILED: Check market_gym.py implementation")
print("=" * 70)
